In [ ]:
# Save preprocessed datasets in multiple formats for different use cases

# 1. Save cleaned data (with original values for interpretability)
df_clean_path = 'animal_disease_cleaned.csv'
df.to_csv(df_clean_path, index=False)
print(f"✓ Cleaned dataset saved: {df_clean_path}")

# 2. Save encoded data (with categorical encoding)
df_encoded_path = 'animal_disease_encoded.csv'
df_encoded.to_csv(df_encoded_path, index=False)
print(f"✓ Encoded dataset saved: {df_encoded_path}")

# 3. Save scaled data (with feature scaling for ML models)
df_scaled_path = 'animal_disease_scaled.csv'
df_processed.to_csv(df_scaled_path, index=False)
print(f"✓ Scaled dataset saved: {df_scaled_path}")

print("\n" + "=" * 80)
print("PREPROCESSING SUMMARY")
print("=" * 80)
print(f"""
Dataset Files Created:
1. {df_clean_path} - Cleaned data with type conversions
2. {df_encoded_path} - Encoded data with categorical variables encoded
3. {df_scaled_path} - Scaled data ready for ML models

Dataset Statistics:
- Original rows: 46,001
- Final rows: {len(df_processed)}
- Columns: {len(df_processed.columns)}
- Missing values: {df_processed.isnull().sum().sum()}

Key Preprocessing Steps Applied:
✓ Loaded and inspected data structure
✓ Handled missing values (if any)
✓ Removed duplicate rows
✓ Converted Duration to numeric format
✓ Converted Yes/No to binary (1/0)
✓ Label encoded target and key features
✓ One-hot encoded Species
✓ Applied StandardScaler normalization
✓ Detected outliers (IQR method)
✓ Validated data quality

Next Steps:
1. Perform Exploratory Data Analysis (EDA)
2. Create visualizations for data insights
3. Engineer additional features if needed
4. Prepare data splits (train/test/validation)
5. Build and train ML models
""")

## 9. Save Preprocessed Data

Save the cleaned and preprocessed dataset for future analysis.

In [ ]:
print("=" * 80)
print("FINAL DATA VALIDATION AND QUALITY CHECKS")
print("=" * 80)

# 1. Check for remaining missing values
print("\n1. MISSING VALUES CHECK")
print("-" * 80)
missing_check = df_processed.isnull().sum().sum()
if missing_check == 0:
    print("✓ PASS: No missing values remaining")
else:
    print(f"✗ FAIL: {missing_check} missing values found")

# 2. Verify data types
print("\n2. DATA TYPES VERIFICATION")
print("-" * 80)
print(f"✓ Total columns: {len(df_processed.columns)}")
print(f"✓ Numerical columns: {df_processed.select_dtypes(include=['int64', 'float64']).shape[1]}")
print(f"✓ Categorical columns (object): {df_processed.select_dtypes(include=['object']).shape[1]}")

# 3. Check value ranges
print("\n3. VALUE RANGES VALIDATION")
print("-" * 80)
print(f"✓ Age range: {df['Age'].min()} - {df['Age'].max()}")
print(f"✓ Weight range: {df['Weight'].min():.2f} - {df['Weight'].max():.2f}")
print(f"✓ Body Temperature range: {df['Body_Temperature'].min()} - {df['Body_Temperature'].max()}")
print(f"✓ Heart Rate range: {df['Heart_Rate'].min()} - {df['Heart_Rate'].max()}")
print(f"✓ Duration range: {df['Duration_Numeric'].min()} - {df['Duration_Numeric'].max()} days")

# 4. Check categorical values
print("\n4. CATEGORICAL VALUES VALIDATION")
print("-" * 80)
print(f"✓ Unique Species: {df['Species'].nunique()}")
print(f"✓ Unique Genders: {df['Gender'].nunique()}")
print(f"✓ Unique Diseases: {df['Disease_Prediction'].nunique()}")
print(f"✓ Binary columns (0/1 only): {all(df[col].isin([0, 1]).all() for col in binary_columns)}")

# 5. Shape summary
print("\n5. DATASET SHAPE SUMMARY")
print("-" * 80)
print(f"Original dataset shape: (46001, 24)")
print(f"Current dataset shape: {df_processed.shape}")
print(f"Rows removed: {46001 - df_processed.shape[0]}")

# 6. Summary statistics
print("\n6. FINAL DATASET STATISTICS")
print("-" * 80)
print(df_processed[['Age', 'Weight', 'Body_Temperature', 'Heart_Rate']].describe())

print("\n" + "=" * 80)
print("✓ DATA PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 80)
print("\nDataset is now ready for:")
print("  - Exploratory Data Analysis (EDA)")
print("  - Feature Engineering")
print("  - Machine Learning Model Development")

## 8. Data Validation and Quality Checks

Perform final validation checks to ensure data quality and readiness for analysis.

In [ ]:
print("=" * 80)
print("OUTLIER DETECTION ANALYSIS (IQR METHOD)")
print("=" * 80)

# Use original data for outlier detection
df_original = df.copy()

def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detect outliers in numerical columns
outlier_summary = {}
for col in ['Age', 'Weight', 'Body_Temperature', 'Heart_Rate']:
    outliers, lower, upper = detect_outliers_iqr(df_original, col)
    outlier_summary[col] = {
        'count': len(outliers),
        'percentage': (len(outliers) / len(df_original)) * 100,
        'lower_bound': lower,
        'upper_bound': upper
    }
    print(f"\n{col}:")
    print(f"  Outliers found: {len(outliers)} ({(len(outliers)/len(df_original))*100:.2f}%)")
    print(f"  Lower bound: {lower:.2f}, Upper bound: {upper:.2f}")

# Visualize outliers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(['Age', 'Weight', 'Body_Temperature', 'Heart_Rate']):
    axes[idx].boxplot(df_original[col], vert=True)
    axes[idx].set_title(f'Boxplot: {col}')
    axes[idx].set_ylabel(col)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("OUTLIER TREATMENT DECISION")
print("=" * 80)
print("Outliers found but kept in dataset (no removal applied)")
print("Reason: In disease data, extreme values may represent severe cases")
print("Alternative: Could implement capping/winsorization if needed")

## 7. Outlier Detection and Treatment

Identify outliers in numerical columns using IQR and Z-score methods.

In [ ]:
print("=" * 80)
print("FEATURE SCALING - BEFORE AND AFTER")
print("=" * 80)

# Define numerical columns to scale
features_to_scale = ['Age', 'Weight', 'Body_Temperature', 'Heart_Rate', 'Duration_Numeric']

# Display original statistics
print("\nORIGINAL NUMERICAL FEATURES STATISTICS")
print(df_encoded[features_to_scale].describe())

# StandardScaler: Standardize features (mean=0, std=1)
scaler_standard = StandardScaler()
df_scaled_standard = df_encoded.copy()
df_scaled_standard[features_to_scale] = scaler_standard.fit_transform(df_encoded[features_to_scale])

print("\n" + "=" * 80)
print("STANDARDIZED FEATURES STATISTICS (StandardScaler)")
print("=" * 80)
print(df_scaled_standard[features_to_scale].describe())

# MinMaxScaler: Normalize features (range 0-1)
scaler_minmax = MinMaxScaler()
df_scaled_minmax = df_encoded.copy()
df_scaled_minmax[features_to_scale] = scaler_minmax.fit_transform(df_encoded[features_to_scale])

print("\n" + "=" * 80)
print("NORMALIZED FEATURES STATISTICS (MinMaxScaler)")
print("=" * 80)
print(df_scaled_minmax[features_to_scale].describe())

# We'll use StandardScaler for further analysis
df_processed = df_scaled_standard.copy()
print("\nUsing StandardScaler for further analysis")

## 6. Feature Scaling and Normalization

Normalize or standardize numerical features to ensure comparable scales.

In [ ]:
# Create a copy for encoding to preserve original data
df_encoded = df.copy()

print("=" * 80)
print("CATEGORICAL ENCODING")
print("=" * 80)

# Label Encoding for target variable and high-cardinality features
label_encoder_dict = {}
label_encode_cols = ['Gender', 'Disease_Prediction']

for col in label_encode_cols:
    le = LabelEncoder()
    df_encoded[col + '_Encoded'] = le.fit_transform(df_encoded[col])
    label_encoder_dict[col] = le
    print(f"\nLabel Encoded '{col}':")
    for i, class_label in enumerate(le.classes_):
        print(f"  {class_label}: {i}")

# One-Hot Encoding for lower cardinality features
onehot_cols = ['Species']
df_encoded = pd.get_dummies(df_encoded, columns=onehot_cols, drop_first=False, prefix=onehot_cols)
print(f"\nOne-Hot Encoded '{onehot_cols}' columns")

# Display sample of encoded data
print("\n" + "=" * 80)
print("SAMPLE OF ENCODED DATASET")
print("=" * 80)
print(df_encoded.head())

## 5. Encode Categorical Variables

Encode categorical columns using Label Encoding or One-Hot Encoding based on cardinality.

In [ ]:
# Convert Duration column from "X days" format to numeric
print("=" * 80)
print("DATA TYPE CONVERSION")
print("=" * 80)

# Extract numeric value from Duration column (e.g., "2 days" -> 2)
df['Duration_Numeric'] = df['Duration'].str.extract('(\d+)').astype(float)
print("Converted Duration to numeric format (days)")

# Convert Yes/No columns to binary (1/0)
binary_columns = ['Appetite_Loss', 'Vomiting', 'Diarrhea', 'Coughing', 
                  'Labored_Breathing', 'Lameness', 'Skin_Lesions', 
                  'Nasal_Discharge', 'Eye_Discharge']

for col in binary_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
    
print(f"Converted {len(binary_columns)} Yes/No columns to binary (1/0)")

# Identify categorical and numerical columns
numerical_cols = ['Age', 'Weight', 'Body_Temperature', 'Heart_Rate', 'Duration_Numeric']
categorical_cols = ['Species', 'Breed', 'Gender', 'Symptom_1', 'Symptom_2', 
                   'Symptom_3', 'Symptom_4', 'Disease_Prediction', 'Treatment']

print("\nNumerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

# Display updated data types
print("\n" + "=" * 80)
print("UPDATED DATA TYPES")
print("=" * 80)
print(df.dtypes)

## 4. Data Type Conversion

Convert columns to appropriate data types and handle specific formats.

In [ ]:
# Check for duplicates
print("=" * 80)
print("DUPLICATE VALUES ANALYSIS")
print("=" * 80)
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print(f"\nRemoving {duplicate_count} duplicate rows...")
    df.drop_duplicates(inplace=True)
    print(f"Dataset shape after removing duplicates: {df.shape}")
else:
    print("No duplicate rows found in the dataset.")

print(f"\nFinal dataset shape after duplicate removal: {df.shape}")

## 3. Remove Duplicates

Identify and remove duplicate rows to ensure data integrity.

In [ ]:
# Check for missing values
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_values.values,
    'Percentage': missing_percent.values
})
print(missing_df)

# If there are missing values, handle them
if missing_values.sum() > 0:
    print("\n" + "=" * 80)
    print("HANDLING MISSING VALUES")
    print("=" * 80)
    # Strategy: Display columns with missing values for decision making
    columns_with_missing = missing_df[missing_df['Missing_Count'] > 0]
    print(f"\nColumns with missing values:\n{columns_with_missing}")
    
    # Example: Fill numerical columns with median, categorical with mode
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if df[col].dtype in ['float64', 'int64']:
                df[col].fillna(df[col].median(), inplace=True)
                print(f"Filled {col} with median")
            else:
                df[col].fillna(df[col].mode()[0], inplace=True)
                print(f"Filled {col} with mode")
else:
    print("\nNo missing values found in the dataset.")

## 2. Handle Missing Values

Check for missing values and implement appropriate handling strategies.

In [ ]:
# Load the dataset
file_path = '--- animal_disease_world_dataset_updated.csv'
df = pd.read_csv(file_path)

# Display basic information about the dataset
print("=" * 80)
print("DATASET SHAPE AND BASIC INFO")
print("=" * 80)
print(f"Dataset Shape: {df.shape}")
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")
print("\n" + "=" * 80)
print("COLUMN NAMES AND DATA TYPES")
print("=" * 80)
print(df.dtypes)
print("\n" + "=" * 80)
print("FIRST FEW ROWS")
print("=" * 80)
print(df.head())

print("\n" + "=" * 80)
print("DATASET STATISTICS")
print("=" * 80)
print(df.describe())

## 1. Load and Inspect the Dataset

Load the CSV file and examine its structure, data types, and basic statistics.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

# Animal Disease World Dataset - Initial Data Preprocessing

This notebook covers comprehensive data preprocessing for the Animal Disease World Dataset, including:
- Loading and inspecting data
- Handling missing values
- Removing duplicates
- Data type conversion
- Categorical encoding
- Feature scaling
- Outlier detection
- Data validation